<a href="https://colab.research.google.com/github/one-2730/ESSA-25-1/blob/Assignment/ESAA_OB_0505_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#2장 실습환경 설정과 파이토치 기초

##2.2 파이토치 기초 문법

In [1]:
###2.2.1 텐서 다루기

#텐서 생성 및 변환

import torch
print(torch.tensor([[1, 2], [3, 4]])) #2차원 형태의 텐서 생성
print(torch.tensor([[1, 2], [3, 4]], device='cuda:0')) #GPU에 텐서 생성
print(torch.tensor([[1, 2], [3, 4]], dtype=torch.float64))

tensor([[1, 2],
        [3, 4]])
tensor([[1, 2],
        [3, 4]], device='cuda:0')
tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)


In [2]:
temp = torch.tensor([[1, 2], [3, 4]])
print(temp.numpy()) #텐서를 ndarray로 변환

temp = torch.tensor([[1, 2], [3, 4]], device='cuda:0')
print(temp.to('cpu').numpy()) #GPU상의 텐서를 CPU의 텐서로 변환한 후 ndarray로 변환

[[1 2]
 [3 4]]
[[1 2]
 [3 4]]


In [5]:
#텐서의 인덱스 조작

temp = torch.FloatTensor([1, 2, 3, 4, 5, 6, 7])
print(temp[0], temp[-1], temp[-1]) #인덱스로 접근
print(temp[2:5], temp[4:-1]) #슬라이스로 접근

tensor(1.) tensor(7.) tensor(7.)
tensor([3., 4., 5.]) tensor([5., 6.])


In [4]:
#텐서 연산 및 차원 조작

#연산
v = torch.tensor([[1, 2, 3]]) #길이가 3인 벡터 생성
w = torch.tensor([3, 4, 6])
print(w - v) #길이가 같은 벡터 간 뺄셈 연산

tensor([[2, 2, 3]])


In [6]:
#차원 조작
temp = torch.tensor([[1, 2], [3, 4]])

print(temp.shape)
print(temp.view(4, 1))
print(temp.view(-1))
print(temp.view(1, -1))
print(temp.view(-1, 1))

torch.Size([2, 2])
tensor([[1],
        [2],
        [3],
        [4]])
tensor([1, 2, 3, 4])
tensor([[1, 2, 3, 4]])
tensor([[1],
        [2],
        [3],
        [4]])


In [ ]:
###2.2.2 데이터 준비

import pandas as pd
data = pd.read_csv('.../file.csv')

x = torch.from_numpy(data['x'].values).unsqueeze(dim=1).float()
y = torch.from_numpy(data['y'].values).unsqueeze(dim=1).float()

#커스텀 데이터셋 만들기

import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class CustomDataset(Dataset):
  def __init__(self, csv_file):
    self.label = pd.read_csv(csv_file)

  def __len__(self):
    return len(self.label)
  def __getitem__(self, idx):
    sample =torch.tensor(self.label.iloc[idx, 0:3]).int()
    label = torch.tensor(self.label.iloc[idx, 3]).int()
    return sample, label

  tensor_dataset = CustomDataset('.../file.csv')
  dataset = DataLoader(tensor_dataset, batch_size=4, shuffle=True)

In [11]:
#파이토치에서 제공하는 데이터셋 사용

import torchvision.transforms as transforms

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, ), (1.0, ))
])

from torchvision.datasets import MNIST
import requests

train_dataset = MNIST(root='', transform=mnist_transform, train=True, download=True)
valid_dataset = MNIST(root='', transform=mnist_transform, train=False, download=True)
test_dataset = MNIST(root='', transform=mnist_transform, train=False, download=True)

In [15]:
###2.2.3 모델 정의
import torch.nn as nn
#단순 신경망을 정의
model = nn.Linear(in_features=1, out_features=1, bias=True)

#nn.Module()을 상속하여 정의
class MLP(nn.Module):
  def __init__(self, inputs):
    super(MLP, self).__init__()
    self.layer = Linear(inputs, 1)
    self.activation = Sigmoid()

  def forward(self, X):
    X = self.layer(X)
    X = self.activation(X)
    return X

In [19]:
#Sequential 신경망을 정의

import torch.nn as nn

class MLP(nn.Module):
  def __init__(self):
    super(MLP, self).__init__()
    self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=64, kernel_size=5),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2)
    )

    self.layer2 = nn.Sequential(
        nn.Conv2d(in_channels=64, out_channels=30, kernel_size=5),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2)
    )

    self.layer3 = nn.Sequential(
        nn.Linear(in_features=30*5*5, out_features=10, bias=True),
        nn.ReLU(inplace=True),
    )

    def forward(self, x):
      x = self.layer1(x)
      x = self.layer2(x)
      x = x.view(x.shape[0], -1)
      x = self.layer3(x)
      return x

model = MLP()

print('printing children')
print(list(model.children()))
print(list(model.modules()))

printing children
[Sequential(
  (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
), Sequential(
  (0): Conv2d(64, 30, kernel_size=(5, 5), stride=(1, 1))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
), Sequential(
  (0): Linear(in_features=750, out_features=10, bias=True)
  (1): ReLU(inplace=True)
)]
[MLP(
  (layer1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(64, 30, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer3): Sequential(
    (0): Linear(in_features=750, out_features=10, bias=True)
    (1): ReLU(inplace=True)
  )
),

In [20]:
#함수로 신경망 정의

def MLP(in_features=1, hidden_features=20, out_features=1):
  hidden = nn.Linear(in_features=in_features, out_features=hidden_features, bias=True)
  activation = nn.ReLU()
  output = nn.Linear(in_features=hidden_features, out_features=out_features, bias=True)
  net = nn.Sequential(hidden, activation, output)
  return net

In [ ]:
### 2.2.5 모델 훈련

for epoch in range(100):
  yhat = model(x_train)
  loss = criterion(yhat, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

In [23]:
### 2.2.6 모델 평가

!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [24]:
### 2.2.6 모델 평가

#함수 이용
import torch
import torchmetrics

preds = torch.randn(10, 5).softmax(dim=-1)
target = torch.randint(5, (10, ))
acc = torchmetrics.functional.accuracy(preds, target, task="multiclass", num_classes=5)

In [ ]:
#모듈 이용
import torch
import torchmetrics
metric = torchmetrics.Accuracy()

n_batches = 10
for i in range(n_batches):
  preds = torch.randn(10, 5).softmax(dim=-1)
  target = torch.randint(5, (10, ))

  acc = metric(preds, target)
  print(f"Accuracy on batch {i}: {acc}")

acc = metric.compute()
print(f"Accuracy on all data: {acc}")

In [27]:
###2.2.7 훈련 과정 모니터링

!pip install tensorboard

In [ ]:
import torch
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('')

for epoch in range(num_epochs):
  model.train()
  batch_loss = 0.0

  for i, (x, y) in enumerate(dataloader):
    x, y = x.to(device).float(), y.to(device).float()
    outputs = model(x)
    loss = criterion(outputs, y)
    writer.add_scalar('Loss', loss, epoch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

writer.close()

In [ ]:
tensorboard --logdir= ''--port=6006

텐서는 뭘까요?


1차원이면 벡터


2차원이면 행렬


3차원이면 텐서라고 한다네요




<br>


파이토치를 왜 쓸까요? (vs텐서플로)
- 단순함(효율적이고 직관적이래요)
- 성능(낮은 CPU 사용, 속도가 빠름)
- 직관적인 인터페이스

파이토치 API
- torch: GPU를 지원하는 텐서 패키지
- torch.autograd: 자동 미분 패키지
- torch.nn: 신경망 구축 및 훈련 패키지
- torch.multiprocessing: 파이썬 멀티프로세싱 패키지
- torch.utils: DataLoader 및 기타 유틸리티를 제공하는 패키지

모델과 모듈은 무엇이 다를까요?
- 계층(layer): 모듈 또는 모듈을 구성하는 한 개의 계층으로 합성곱층, 선형 계층 등이 있습니다
- 모듈(module): 한 개 이상의 계층이 모여서 구성된 것으로, 모듈이 모여 새로운 모듈를 만들 수도 있습니다.
- 모델(model): 최종적으로 원하는 네트워크로, 한 개의 모듈이 모델이 될 수도 있습니다.

모델 학습 전에 정의해야 할 파라미터는 뭐가 있을까요?

- 손실 함수
  - BCELoss: 이진분류
  - CrossEntropyLoss: 다중 클래스 분류
  - MSELoss: 회귀 모델
- 옵티마이저: 모델의 업데이트 방법 결정
- 학습률 스케줄러: 미리 지정한 횟수의 에포크를 지날 때마다 학습률을 감소시켜줌. 초기에는 빠른 학습하다가 나중 가서 learning rate를 줄여서 최적을 잘 찾을 수 있어요.